# Gemma 4 Trust & Safety Context Pruning Runner

Use this notebook inside Kaggle for Gemma 4 Trust & Safety research experiments.
It runs the ACPA pipeline, creates benchmark artifacts, and writes JSONL output.

Expected Kaggle setup:

1. Attach the Agentic Eval dataset to this notebook.
2. Add your Gemma/Gemini API key through Kaggle Secrets using label `GEMINI_API_KEY`.
3. Turn Kaggle notebook **Internet** on.
4. Click **Run All**.

The first code cell uses the direct GitHub clone/bootstrap path: it clones the main branch into `/kaggle/working/context-pruning`, adds `/kaggle/working/context-pruning/src` to `sys.path`, and verifies that `acpa_gemma` is importable.

Do not paste API keys into public notebook cells or commit them to Git.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Direct GitHub bootstrap for Kaggle "Run All".
# Requires Kaggle notebook Internet = On.
REPO_URL = 'https://github.com/joyjeni/context-pruning.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/kaggle/working/context-pruning') if Path('/kaggle/working').exists() else Path.cwd() / 'context-pruning'
SRC_ROOT = REPO_ROOT / 'src'

if not (SRC_ROOT / 'acpa_gemma').exists():
    if REPO_ROOT.exists():
        print('Removing stale repo directory:', REPO_ROOT)
        shutil.rmtree(REPO_ROOT)
    print('Cloning repository branch:', REPO_BRANCH)
    try:
        subprocess.run(
            [
                'git',
                'clone',
                '--depth',
                '1',
                '--branch',
                REPO_BRANCH,
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
    except Exception as exc:
        raise RuntimeError(
            'Could not clone the repository. Turn Kaggle Notebook Internet ON, '
            'then rerun from the first cell. If Internet is disabled, upload the '
            'repo as a Kaggle dataset and set REPO_ROOT to that dataset path.'
        ) from exc

if not (SRC_ROOT / 'acpa_gemma').exists():
    raise RuntimeError(f'acpa_gemma source package not found under {SRC_ROOT}')

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT exists =', SRC_ROOT.exists())
print('acpa_gemma exists =', (SRC_ROOT / 'acpa_gemma').exists())
print('Kaggle input exists =', Path('/kaggle/input').exists())

import acpa_gemma
print('Imported acpa_gemma from:', acpa_gemma.__file__)


## Dependency install

Run this cell as normal Python. Do **not** type `pip install ...` without a leading `!` in a Kaggle code cell; that causes `SyntaxError`. The cell below uses `python -m pip` through `subprocess`, so it works as valid Python. Internet must be enabled in the notebook settings if packages are missing.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'google.genai': 'google-genai',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All required packages are already installed.')


## Locate Agentic Eval dataset

This cell prints all attached Kaggle dataset directories, counts supported files, and selects the directory with the most CSV/JSON/JSONL/NDJSON/Parquet files. If it chooses the wrong directory, set `MANUAL_DATASET_DIR` in the code cell below to the attached dataset directory before continuing.

If this cell reports `/kaggle/input` with `total_files=0`, no Kaggle input data is attached to the notebook yet. This is not a code error. Open the notebook right sidebar, click **Add data** / **Input**, attach or upload the Agentic Eval records, then rerun this cell.

If Kaggle only shows a competition note such as `/kaggle/input/competitions/.../NOTE.md`, the Agentic Eval records are not attached yet. The dry-run and benchmark cells can still continue with the built-in demo record, but the real Gemma run requires an attached dataset with CSV/JSON/JSONL/NDJSON/Parquet files.


In [ ]:
from acpa_gemma.data import dataset_diagnostics, discover_dataset_files, format_dataset_diagnostics, load_agentic_eval_dataset

# Optional manual override. Example:
# MANUAL_DATASET_DIR = '/kaggle/input/my-agentic-eval-dataset'
MANUAL_DATASET_DIR = ''

input_root = Path('/kaggle/input')
if MANUAL_DATASET_DIR:
    DATASET_DIR = Path(MANUAL_DATASET_DIR)
elif input_root.exists():
    candidates = sorted([p for p in input_root.iterdir() if p.is_dir()])
    if not candidates:
        candidates = [input_root]
    print('Available Kaggle input directories:')
    for candidate in candidates:
        diagnostics = dataset_diagnostics(candidate, max_files=5)
        print('\n---')
        print(candidate)
        print('supported_files =', diagnostics['supported_files'])
        for file_path in diagnostics['sample_supported_files']:
            print('  supported:', file_path)

    DATASET_DIR = max(candidates, key=lambda path: dataset_diagnostics(path)['supported_files'])
else:
    DATASET_DIR = Path('demo-missing-kaggle-input')

print('\nSelected DATASET_DIR =', DATASET_DIR)
selected_diagnostics = dataset_diagnostics(DATASET_DIR, max_files=20)
print(format_dataset_diagnostics(selected_diagnostics))

preview_records = load_agentic_eval_dataset(DATASET_DIR, sample_size=3)
HAS_AGENTIC_EVAL_DATA = bool(preview_records)
print('Preview records loaded =', len(preview_records))
for record in preview_records:
    print('-', record.record_id, 'source=', record.source_path, 'chars=', len(record.combined_text()))

if not HAS_AGENTIC_EVAL_DATA:
    print('\nSTOP: No Agentic Eval records were loaded from DATASET_DIR.')
    if selected_diagnostics['exists'] and selected_diagnostics['total_files'] == 0:
        print('/kaggle/input is empty, so no Kaggle dataset/input is attached to this notebook.')
        print('Open the notebook right sidebar, click Add data/Input, attach or upload the')
        print('Agentic Eval files, then rerun this cell. The files must be CSV, JSON,')
        print('JSONL, NDJSON, or Parquet.')
    elif selected_diagnostics['supported_files'] == 0:
        print('Files are attached, but none have a supported suffix.')
        print('Set MANUAL_DATASET_DIR to the folder containing CSV/JSON/JSONL/NDJSON/Parquet records,')
        print('or upload the dataset in one of those formats.')
    else:
        print('Supported files were found, but no rows could be parsed. Check the file schema/content.')
    print('Dry-run and benchmark cells can still use the built-in demo record,')
    print('but the real Gemma run requires attached Agentic Eval records.')


## Add and verify your Google API key using Kaggle Secrets

Write the secret in the Kaggle UI, not in this notebook:

1. Open the notebook right sidebar.
2. Click **Add-ons**.
3. Click **Secrets**.
4. Add a secret with label `GEMINI_API_KEY` and value equal to your Google AI Studio / Gemini API key.
5. Turn on notebook access for that secret.

The code cell below is where you verify the secret and write `/kaggle/working/configs/secrets.toml`, which is what the pipeline reads. It prints only `True`/`False`, never the key itself.


In [ ]:
config_dir = Path('/kaggle/working/configs') if Path('/kaggle/working').exists() else REPO_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

# This label must exactly match the label you added in Kaggle Add-ons -> Secrets.
secret_label = 'GEMINI_API_KEY'
api_key = ''
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret(secret_label)
except Exception as exc:
    print(f'Could not load Kaggle Secret {secret_label!r}: {exc}')

print('API key loaded from Kaggle Secrets =', bool(api_key))

app_toml = f'''
[gemma]
model = "gemma-4-26b-a4b-it"
temperature = 0.2
top_p = 0.9
max_output_tokens = 2048

[data]
input_dir = "{DATASET_DIR}"
sample_size = 0

[pruning]
alpha = 1.5
beta = 1.0
gamma = 0.5
delta = 10.0
prune_ratio = 0.45
cache_threshold = 2
priority_boost = 1.5

[output]
path = "/kaggle/working/results.jsonl"
'''.strip() + "\n"

secrets_toml = f'''
[gemma]
api_key = "{api_key}"
'''.strip() + "\n"

(config_dir / 'app.toml').write_text(app_toml, encoding='utf-8')
(config_dir / 'secrets.toml').write_text(secrets_toml, encoding='utf-8')

print('Wrote', config_dir / 'app.toml')
print('Wrote', config_dir / 'secrets.toml')
print('API key present =', bool(api_key))


## Dry-run pipeline test, no API key required

This verifies dataset loading, context construction, ACPA pruning, and JSONL output.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/dry_run.jsonl',
    '--sample-size', '3',
    '--dry-run',
])

print(Path('/kaggle/working/dry_run.jsonl').read_text(encoding='utf-8')[:2000])


## Benchmark ACPA vs baseline algorithms, no API key required

Creates per-record CSV, aggregate CSV, and a Markdown report.


In [ ]:
from acpa_gemma.benchmark import main as benchmark_main

benchmark_main([
    '--input', str(DATASET_DIR),
    '--sample-size', '100',
    '--details-output', '/kaggle/working/benchmark_details.csv',
    '--summary-output', '/kaggle/working/benchmark_summary.csv',
    '--report-output', '/kaggle/working/benchmark_report.md',
])

print(Path('/kaggle/working/benchmark_report.md').read_text(encoding='utf-8'))


## Real Gemma 4 run

Run this after `API key present = True`. Start with a small sample, then remove `--sample-size` for the full run.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

if not api_key:
    raise RuntimeError('Add GEMINI_API_KEY/GEMMA_API_KEY in Kaggle Secrets or paste api_key in the config cell first.')
if not HAS_AGENTIC_EVAL_DATA:
    raise RuntimeError(
        'No Agentic Eval records are attached. Add the dataset in Kaggle Add data, '
        'or set DATASET_DIR to a directory containing CSV/JSON/JSONL/NDJSON/Parquet files, '
        'then rerun the dataset locator cell before calling the real Gemma API.'
    )

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/results_sample.jsonl',
    '--sample-size', '10',
])

print(Path('/kaggle/working/results_sample.jsonl').read_text(encoding='utf-8')[:3000])


## Full results run

Uncomment this cell when the sample run looks good.


In [ ]:
# pipeline_main([
#     '--config', str(config_dir / 'app.toml'),
#     '--secrets', str(config_dir / 'secrets.toml'),
#     '--input', str(DATASET_DIR),
#     '--output', '/kaggle/working/results.jsonl',
# ])


## Output files

Attach these files to your research report, hackathon demo, or publication appendix as needed:

- `/kaggle/working/results.jsonl`
- `/kaggle/working/results_sample.jsonl`
- `/kaggle/working/benchmark_report.md`
- `/kaggle/working/benchmark_summary.csv`
- `/kaggle/working/benchmark_details.csv`
